# 🏆 پیش‌بینی جام جهانی ۲۰۲۶ (پروژه پایان دوره Edox)
در این نوتبوک، تمام مراحل پروژه را برای ارائه مرور می‌کنیم:
۱. **آپدیت زنده داده‌ها (Live Update)**
۲. **تحلیل داده‌های تاریخی (EDA)**
۳. **عملکرد مدل روی جام جهانی ۲۰۲۲ (دوره قبل)**
۴. **عملکرد مدل روی بازی‌های جام جهانی ۲۰۲۶**
۵. **پیش‌بینی کامل مسابقات آینده**
۶. **شبیه‌ساز زنده مسابقات (Live Simulator)**

In [ ]:
# نصب کتابخانه‌های مورد نیاز (فقط کافیست این سلول را اجرا کنید)
%pip install -q pandas numpy matplotlib scikit-learn plotly joblib ipython xgboost lightgbm openpyxl requests beautifulsoup4

In [ ]:
# وارد کردن کتابخانه‌ها
import pandas as pd
import numpy as np
import joblib
import requests
from bs4 import BeautifulSoup
import re
from pathlib import Path
from IPython.display import Image, display, Markdown, HTML
import warnings
import os
warnings.filterwarnings('ignore')

# تنظیم مسیرها به صورت هوشمند (چه نوتبوک در فولدر notebooks باشد چه در ریشه پروژه)
current_dir = Path(os.getcwd())
if current_dir.name == 'notebooks':
    BASE_DIR = Path('../')
else:
    BASE_DIR = Path('.')

DATA_DIR = BASE_DIR / "data" / "processed"
RAW_DIR = BASE_DIR / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "output"
MODELS_DIR = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"

model_dataset = None
model_goals_a = None
model_goals_b = None
features_to_drop = []

## ۱. آپدیت زنده داده‌ها (Live Data Pipeline)
این بخش نتایج بازی‌های جدید را (چه به صورت دستی از فایل `live_new_matches.csv` و چه به صورت اتوماتیک از اینترنت) دریافت می‌کند و بدون دست زدن به دیتابیس‌های اصلی شما، یک نسخه زنده به نام `model_dataset_live.csv` برای پیش‌بینی می‌سازد.

In [ ]:
def clean_team_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    team_map = {
        "United States": "USA", "U.S.A.": "USA", "USMNT": "USA",
        "Iran": "IR Iran", "South Korea": "Korea Republic",
        "Turkey": "Türkiye", "Cape Verde": "Cabo Verde",
        "Ivory Coast": "Côte d'Ivoire", "Cote d'Ivoire": "Côte d'Ivoire",
        "Czech Republic": "Czechia", "England": "England"
    }
    return team_map.get(name, name)

def result_label(home_score, away_score):
    if home_score > away_score:
        return "team_a_win"
    if home_score < away_score:
        return "team_b_win"
    return "draw"

print("🔄 در حال بررسی داده‌های زنده...")
scraped_matches = []
try:
    # اسکرپ اتوماتیک از ویکی‌پدیا
    response = requests.get("https://en.wikipedia.org/wiki/2026_FIFA_World_Cup", timeout=5)
    soup = BeautifulSoup(response.content, "html.parser")
    boxes = soup.find_all("div", class_="footballbox")
    for box in boxes:
        home = box.find("th", class_="fhome")
        score = box.find("th", class_="fscore")
        away = box.find("th", class_="faway")
        if home and score and away:
            s_str = score.get_text(strip=True)
            if 'v' not in s_str.lower() and any(c.isdigit() for c in s_str):
                sp = s_str.split('–') if '–' in s_str else s_str.split('-')
                if len(sp) == 2:
                    try:
                        scraped_matches.append({
                            "date": pd.to_datetime("2026-06-11"),
                            "home_team": home.get_text(strip=True),
                            "away_team": away.get_text(strip=True),
                            "home_score": int(sp[0].strip()),
                            "away_score": int(re.sub(r'\D', '', sp[1].strip()[:2])),
                            "tournament": "FIFA World Cup",
                            "country": "United States",
                            "neutral": True
                        })
                    except Exception:
                        pass
except Exception:
    print("⚠️ اسکرپر نتوانست وصل شود. مشکلی نیست، از فایل دستی استفاده می‌کنیم.")

df_scraped = pd.DataFrame(scraped_matches)
if not df_scraped.empty:
    df_scraped['home_team'] = df_scraped['home_team'].apply(clean_team_name)
    df_scraped['away_team'] = df_scraped['away_team'].apply(clean_team_name)

df_manual = pd.DataFrame()
if (RAW_DIR / "live_new_matches.csv").exists():
    df_manual = pd.read_csv(RAW_DIR / "live_new_matches.csv")

df_new = pd.concat([df_manual, df_scraped]).drop_duplicates(subset=['home_team', 'away_team']) if not df_scraped.empty or not df_manual.empty else pd.DataFrame()

# ادغام با داده‌های قبلی در یک فایل جدید (Live)
base_dataset = pd.read_csv(DATA_DIR / "model_dataset.csv")
if not df_new.empty:
    print(f"✅ {len(df_new)} بازی جدید برای آپدیت پیدا شد!")
    new_rows = []
    for _, row in df_new.iterrows():
        t1, t2 = row['home_team'], row['away_team']
        t1_stats = base_dataset[base_dataset['team_a'] == t1].iloc[-1:] if len(base_dataset[base_dataset['team_a'] == t1]) > 0 else None
        t2_stats = base_dataset[base_dataset['team_b'] == t2].iloc[-1:] if len(base_dataset[base_dataset['team_b'] == t2]) > 0 else None
        if t1_stats is not None and t2_stats is not None:
            nr = t1_stats.copy()
            nr['date'] = pd.to_datetime(row['date']).strftime('%Y-%m-%d')
            nr['team_b'], nr['team_a_goals'], nr['team_b_goals'] = t2, row['home_score'], row['away_score']
            nr['result'] = result_label(row['home_score'], row['away_score'])
            nr['tournament'], nr['neutral'] = row['tournament'], row['neutral']
            for col in nr.columns:
                if 'team_b' in col and col not in ['team_b', 'team_b_goals']:
                    nr[col] = t2_stats[col].values[0]
            new_rows.append(nr)
    if new_rows:
        updated = pd.concat([base_dataset, pd.concat(new_rows)]).reset_index(drop=True)
        updated.to_csv(DATA_DIR / "model_dataset_live.csv", index=False)
        
        # آپدیت فایل مستقل جام جهانی 2026
        wc2026 = updated[(updated['tournament'].isin(['World Cup', 'FIFA World Cup'])) & (pd.to_datetime(updated['date'].astype(str).str.split().str[0]).dt.year == 2026)]
        wc2026.to_csv(DATA_DIR / 'wc2026.csv', index=False)

        display(Markdown("<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>دیتاست با موفقیت به صورت زنده آپدیت شد (model_dataset_live.csv)!</b></div>"))
else:
    print("هیچ دیتای جدیدی نبود. از همان دیتای قبلی استفاده می‌کنیم.")
    base_dataset.to_csv(DATA_DIR / "model_dataset_live.csv", index=False)


## ۲. تحلیل اکتشافی داده‌ها (EDA)
قبل از مدل‌سازی، داده‌های مسابقات بین‌المللی از ده‌ها سال پیش را بررسی کردیم تا متوجه الگوهای برد و باخت تیم‌ها بشویم.

In [ ]:
try:
    display(Markdown("### توزیع نتایج مسابقات"))
    display(Image(filename=str(FIGURES_DIR / "eda_result_distribution.png"), width=600))
    
    display(Markdown("### روند رشد بازی‌های فوتبال در طول تاریخ"))
    display(Image(filename=str(FIGURES_DIR / "eda_matches_per_year.png"), width=600))
except Exception as e:
    print("نمودار یافت نشد. خطا:", e)

## ۳. ارزیابی مدل و مهم‌ترین ویژگی‌ها (Feature Importance)
مدل ما با استفاده از الگوریتم‌های ماشین لرنینگ آموزش داده شده است. در نمودار زیر می‌بینیم که کدام ویژگی‌های تیم‌ها برای ماشین از همه مهم‌تر بوده‌اند.

In [ ]:
try:
    display(Image(filename=str(FIGURES_DIR / "rf_feature_importance_top20.png"), width=800))
except Exception as e:
    print("نمودار یافت نشد. خطا:", e)

## ۴. اعتبارسنجی مدل با جام جهانی ۲۰۲۲
برای اینکه ثابت کنیم مدل ما چقدر خوب کار می‌کند، تمام مسابقات جام جهانی قبلی (۲۰۲۲ قطر) را که در داده‌ها داشتیم جدا کردیم و به مدل دادیم تا آن‌ها را پیش‌بینی کند!

In [ ]:
try:
    # اینجا از دیتابیس زنده (Live) استفاده می‌کنیم
    model_dataset = pd.read_csv(DATA_DIR / "model_dataset_live.csv")
    model_dataset['date'] = pd.to_datetime(model_dataset['date'].astype(str).str.split().str[0])
    wc2022 = pd.read_csv(DATA_DIR / 'wc2022.csv')
    wc2022['date'] = pd.to_datetime(wc2022['date'])
    
    # بارگذاری مدل‌های رگرسیون برای پیش‌بینی گل
    model_goals_a = joblib.load(MODELS_DIR / "best_team_a_goals_model.joblib")
    model_goals_b = joblib.load(MODELS_DIR / "best_team_b_goals_model.joblib")
    
    features_to_drop = ['team_a', 'team_b', 'date', 'team_a_goals', 'team_b_goals', 'result']
    X_2022 = wc2022.drop(columns=[c for c in features_to_drop if c in wc2022.columns])
    
    wc2022['predicted_goals_a'] = np.round(model_goals_a.predict(X_2022)).astype(int)
    wc2022['predicted_goals_b'] = np.round(model_goals_b.predict(X_2022)).astype(int)
    
    wc2022['predicted_goals_a'] = wc2022['predicted_goals_a'].apply(lambda x: max(0, x))
    wc2022['predicted_goals_b'] = wc2022['predicted_goals_b'].apply(lambda x: max(0, x))
    
    def get_winner(g_a, g_b):
        if g_a > g_b: return "team_a_win"
        elif g_b > g_a: return "team_b_win"
        else: return "draw"
        
    wc2022['predicted_result'] = wc2022.apply(lambda row: get_winner(row['predicted_goals_a'], row['predicted_goals_b']), axis=1)
    wc2022['result_correct'] = wc2022['result'] == wc2022['predicted_result']
    
    acc = wc2022['result_correct'].mean() * 100
    display(Markdown(f"<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>دقت پیش‌بینی نتیجه (برد/مساوی/باخت) در جام جهانی ۲۰۲۲: {acc:.1f}%</b></div>"))
    
    display_df = wc2022[['date', 'team_a', 'team_b', 'team_a_goals', 'team_b_goals', 'predicted_goals_a', 'predicted_goals_b', 'result_correct']].tail(15)
    display_df.columns = ['تاریخ', 'تیم الف', 'تیم ب', 'گل الف (واقعی)', 'گل ب (واقعی)', 'گل الف (پیش‌بینی)', 'گل ب (پیش‌بینی)', 'تشخیص درست؟']
    display(display_df)
    
except Exception:
    import traceback
    print("خطا در بارگذاری داده‌ها:")
    traceback.print_exc()

## ۵. عملکرد مدل روی بازی‌های انجام‌شده‌ی جام جهانی ۲۰۲۶ (تست زنده)
حالا مدل را روی مسابقاتی که تا همین امروز در جام جهانی ۲۰۲۶ برگزار شده ارزیابی می‌کنیم تا ببینیم چقدر در پیش‌بینی واقعیت فعلی موفق بوده است. (این بخش آپدیت‌های زنده را هم در نظر می‌گیرد!)

In [ ]:
try:
    # لود کردن فایل مجزای ۲۰۲۶
    wc2026_actual = pd.read_csv(DATA_DIR / 'wc2026.csv')
    wc2026_actual['date'] = pd.to_datetime(wc2026_actual['date'])
    
    if not wc2026_actual.empty:
        X_2026 = wc2026_actual.drop(columns=[c for c in features_to_drop if c in wc2026_actual.columns])
        wc2026_actual['pred_a'] = np.round(model_goals_a.predict(X_2026)).astype(int)
        wc2026_actual['pred_b'] = np.round(model_goals_b.predict(X_2026)).astype(int)
        wc2026_actual['pred_res'] = wc2026_actual.apply(lambda r: get_winner(r['pred_a'], r['pred_b']), axis=1)
        wc2026_actual['correct'] = wc2026_actual['result'] == wc2026_actual['pred_res']
        
        acc26 = wc2026_actual['correct'].mean() * 100
        display(Markdown(f"<div style='background-color:#cce5ff; padding:10px; border-radius:5px;'><b>دقت تشخیص برنده در {len(wc2026_actual)} بازی برگزار شده ۲۰۲۶ (از ابتدا تاکنون): {acc26:.1f}%</b></div>"))
        display(wc2026_actual[['date', 'team_a', 'team_b', 'team_a_goals', 'team_b_goals', 'pred_a', 'pred_b', 'correct']])
    else:
        print("هنوز هیچ دیتای واقعی برای جام جهانی ۲۰۲۶ در دیتابیس ثبت نشده است.")
except Exception as e:
    print("خطا در محاسبه دقت:", e)



## ۶. شبیه‌ساز زنده مسابقات (Live Simulator)
در کلاس می‌توانید در این سلول نام هر دو تیمی که می‌خواهید را وارد کنید و کد را اجرا کنید تا ماشین لرنینگ فوراً بازی را پیش‌بینی کند!

In [ ]:
def simulate_match(team1, team2):
    if model_dataset is None or model_goals_a is None:
        print("خطا: دیتاست یا مدل‌ها بارگذاری نشده‌اند. لطفاً سلول‌های قبلی را اجرا کنید.")
        return
        
    t1_stats = model_dataset[model_dataset['team_a'] == team1].iloc[-1:] if len(model_dataset[model_dataset['team_a'] == team1]) > 0 else None
    t2_stats = model_dataset[model_dataset['team_b'] == team2].iloc[-1:] if len(model_dataset[model_dataset['team_b'] == team2]) > 0 else None
    
    if t1_stats is not None and t2_stats is not None:
        demo_X = t1_stats.drop(columns=[c for c in features_to_drop if c in t1_stats.columns]).copy()
        
        # تنظیم آمار تیم مهمان
        for col in demo_X.columns:
            if 'team_b' in col and col in t2_stats.columns:
                demo_X[col] = t2_stats[col].values[0]
                
        if 'tournament' in demo_X.columns: demo_X['tournament'] = 'FIFA World Cup'
        if 'neutral' in demo_X.columns: demo_X['neutral'] = True
                
        try:
            goals_a = max(0, int(np.round(model_goals_a.predict(demo_X)[0])))
            goals_b = max(0, int(np.round(model_goals_b.predict(demo_X)[0])))
            
            display(HTML("<h2 style='color:#004085; text-align:center;'>⚽ شبیه‌سازی زنده مسابقه</h2>"))
            display(HTML(f"<h1 style='text-align:center;'>{team1} <span style='color:red;'>{goals_a} - {goals_b}</span> {team2}</h1>"))
            
            if goals_a > goals_b:
                display(HTML(f"<h3 style='color:green; text-align:center;'>🏆 برنده پیش‌بینی شده: {team1}</h3>"))
            elif goals_b > goals_a:
                display(HTML(f"<h3 style='color:green; text-align:center;'>🏆 برنده پیش‌بینی شده: {team2}</h3>"))
            else:
                display(HTML("<h3 style='color:orange; text-align:center;'>🤝 نتیجه: مساوی</h3>"))
        except Exception as e:
            print("ارور در پیش‌بینی:", e)
    else:
        print("اطلاعات یکی از این تیم‌ها در دیتابیس موجود نیست.")

# ===============
# تیم‌های دلخواه خود را اینجا وارد کرده و این سلول را اجرا کنید
# ===============
simulate_match("Germany", "Brazil")

---
## ۶. اجرای داشبورد تعاملی (Streamlit)
با اجرای سلول زیر، داشبورد حرفه‌ای استریم‌لیت شما باز می‌شود. (توجه: این سلول تا زمانی که خودتان متوقفش نکنید در حال اجرا باقی می‌ماند. برای بستن داشبورد، دکمه Stop (مربع) را در بالای نوتبوک بزنید).

In [ ]:
# نصب استریم‌لیت روی این محیط (فقط یک بار نیاز است)\n%pip install streamlit

In [ ]:
# اجرای استریم‌لیت مستقیماً از داخل نوتبوک (در محیط Edox_ML)
!streamlit run ../app/streamlit_app.py